# UpLift — Notebook 05: Threshold Sensitivity Analysis
**Project:** UpLift — Predictive Maintenance for Transit Accessibility Equipment  
**Author:** Nico Meyering, MPA | Equitech Futures CTI 2026  
**GitHub:** [github.com/meyeringn/uplift-transit](https://github.com/meyeringn/uplift-transit)

---

## What This Notebook Does

The decision threshold is the single most consequential operational parameter in UpLift. It determines how many units get flagged, how many real failures get caught, and how many service calls turn out to be unnecessary.

Different agencies have different constraints:
- A small agency with a tight maintenance budget may need a **higher threshold** — fewer flags, higher confidence each flag is real
- An agency under active ADA enforcement may need a **lower threshold** — catch everything, accept more false alarms

This notebook maps out the full trade-off space so each agency can select the threshold that matches their operational reality.

**Outputs:**
- Threshold sensitivity curve (F2, Recall, Precision vs. threshold)
- Cost-benefit analysis at five representative thresholds
- `uplift_threshold_analysis.csv` — full metric table for every threshold value

In [ ]:
# Install dependencies (run once in Google Colab)
# !pip install xgboost imbalanced-learn shap

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
import joblib
import warnings
warnings.filterwarnings('ignore')

from sklearn.metrics import (
    precision_score, recall_score, fbeta_score, confusion_matrix
)

np.random.seed(42)
sns.set_theme(style='whitegrid')
print('Libraries loaded.')

## 1. Load Model and Holdout Data

In [ ]:
model = joblib.load('uplift_xgboost_model.pkl')
X_holdout = np.load('X_holdout.npy')
y_holdout = np.load('y_holdout.npy')
feature_cols = joblib.load('uplift_feature_cols.pkl')

y_prob = model.predict_proba(X_holdout)[:, 1]

print(f'Holdout records: {len(y_holdout)}')
print(f'Actual failure rate: {y_holdout.mean():.1%}')

## 2. Sweep Thresholds from 0.10 to 0.90

In [ ]:
# Cost assumptions (per unit)
COST_FALSE_POSITIVE = 1500   # Unnecessary service call (dollars)
COST_FALSE_NEGATIVE = 50000  # Conservative ADA/reputational cost estimate per missed failure

thresholds = np.arange(0.10, 0.91, 0.01)
results = []

for t in thresholds:
    y_pred = (y_prob >= t).astype(int)
    cm = confusion_matrix(y_holdout, y_pred)
    tn, fp, fn, tp = cm.ravel() if cm.shape == (2,2) else (cm[0,0], 0, 0, 0)

    prec = precision_score(y_holdout, y_pred, zero_division=0)
    rec = recall_score(y_holdout, y_pred, zero_division=0)
    f2 = fbeta_score(y_holdout, y_pred, beta=2, zero_division=0)
    flags = y_pred.sum()
    total_cost = (fp * COST_FALSE_POSITIVE) + (fn * COST_FALSE_NEGATIVE)

    results.append({
        'threshold': round(t, 2),
        'precision': round(prec, 4),
        'recall': round(rec, 4),
        'f2_score': round(f2, 4),
        'true_positive': int(tp),
        'false_positive': int(fp),
        'false_negative': int(fn),
        'true_negative': int(tn),
        'units_flagged': int(flags),
        'estimated_cost_usd': int(total_cost)
    })

results_df = pd.DataFrame(results)
results_df.to_csv('uplift_threshold_analysis.csv', index=False)
print('Saved: uplift_threshold_analysis.csv')
print(f'\nSample rows:')
results_df[results_df['threshold'].isin([0.20, 0.30, 0.40, 0.50, 0.60, 0.70])]

## 3. Sensitivity Curve

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(12, 10))
fig.suptitle('UpLift — Threshold Sensitivity Analysis', fontweight='bold', fontsize=14)

# Top: Precision, Recall, F2 vs threshold
axes[0].plot(results_df['threshold'], results_df['recall'],
             color='#E53935', lw=2.5, label='Recall (catch rate on real failures)')
axes[0].plot(results_df['threshold'], results_df['precision'],
             color='#1565C0', lw=2.5, label='Precision (accuracy of flags)')
axes[0].plot(results_df['threshold'], results_df['f2_score'],
             color='#2E7D32', lw=2.5, linestyle='--', label='F2-Score (primary metric)')
axes[0].axvline(x=0.30, color='black', linestyle=':', lw=1.5, label='UpLift default (0.30)')
axes[0].set_xlabel('Decision Threshold', fontsize=11)
axes[0].set_ylabel('Score', fontsize=11)
axes[0].set_title('Precision, Recall, and F2-Score by Threshold', fontweight='bold')
axes[0].legend()
axes[0].set_xlim([0.10, 0.90])
axes[0].set_ylim([0, 1])

# Bottom: Units flagged vs threshold
ax2 = axes[1]
ax2.fill_between(results_df['threshold'], results_df['units_flagged'],
                 alpha=0.3, color='#FF9800')
ax2.plot(results_df['threshold'], results_df['units_flagged'],
         color='#FF9800', lw=2.5, label='Units flagged for service')
ax2.axvline(x=0.30, color='black', linestyle=':', lw=1.5, label='UpLift default (0.30)')
ax2.set_xlabel('Decision Threshold', fontsize=11)
ax2.set_ylabel('Units Flagged', fontsize=11)
ax2.set_title('Maintenance Workload by Threshold', fontweight='bold')
ax2.legend()
ax2.set_xlim([0.10, 0.90])

plt.tight_layout()
plt.savefig('uplift_threshold_sensitivity.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: uplift_threshold_sensitivity.png')

## 4. Cost-Benefit Comparison at Five Thresholds

In [ ]:
key_thresholds = [0.20, 0.30, 0.40, 0.50, 0.70]
key_df = results_df[results_df['threshold'].isin(key_thresholds)].copy()
key_df['agency_profile'] = [
    'ADA enforcement pressure',
    'UpLift default (recommended)',
    'Balanced',
    'Standard classifier',
    'Tight maintenance budget'
]

fig, ax = plt.subplots(figsize=(12, 6))
x = np.arange(len(key_thresholds))
width = 0.35

fp_costs = key_df['false_positive'] * COST_FALSE_POSITIVE
fn_costs = key_df['false_negative'] * COST_FALSE_NEGATIVE

bars1 = ax.bar(x - width/2, fp_costs, width, label='Cost of False Positives (wasted calls)',
               color='#FF9800', alpha=0.85, edgecolor='white')
bars2 = ax.bar(x + width/2, fn_costs, width, label='Cost of False Negatives (missed failures)',
               color='#E53935', alpha=0.85, edgecolor='white')

ax.set_xlabel('Decision Threshold', fontsize=11)
ax.set_ylabel('Estimated Cost (USD)', fontsize=11)
ax.set_title('UpLift — Cost of Errors by Threshold\n'
             f'False Positive = ${COST_FALSE_POSITIVE:,}/incident | '
             f'False Negative = ${COST_FALSE_NEGATIVE:,}/incident (conservative)',
             fontweight='bold', fontsize=12)
ax.set_xticks(x)
ax.set_xticklabels([f'{t}\n({p})' for t, p in
                    zip(key_df['threshold'], key_df['agency_profile'])], fontsize=9)
ax.yaxis.set_major_formatter(mtick.FuncFormatter(lambda v, _: f'${v:,.0f}'))
ax.legend()
plt.tight_layout()
plt.savefig('uplift_threshold_cost.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: uplift_threshold_cost.png')

print('\nFull comparison table:')
print(key_df[['threshold','agency_profile','recall','precision','f2_score',
              'units_flagged','false_negative','estimated_cost_usd']].to_string(index=False))

## 5. Threshold Selection Guide

| Agency Situation | Recommended Threshold | Rationale |
|---|---|---|
| Active ADA enforcement / legal exposure | 0.20–0.25 | Maximize recall; accept more service calls to minimize missed failures |
| Standard deployment (recommended) | **0.30** | UpLift default; strong recall with manageable workload |
| Moderate budget constraints | 0.40–0.45 | Balanced precision/recall trade-off |
| Severe budget constraints | 0.55–0.65 | High confidence flags only; understand recall will drop significantly |

> **Note:** Threshold selection is an agency decision, not a model decision. UpLift provides the probability scores and the full trade-off curve. The agency selects the operating point that matches its budget, legal exposure, and equity commitments.

➡️ **Next:** Notebook 06 — Agency-Facing Summary Report